## **REDES NEURONALES**

Scikit-learn implementa redes neuronales básicas del tipo Perceptrón Multicapa (MLP) para tareas de aprendizaje supervisado, diseñadas para conjuntos de datos pequeños o medianos.

Características principales

Arquitectura de Feedforward: La información viaja en una sola dirección: desde la capa de entrada, pasa por las capas ocultas y llega a la capa de salida.

Tipos de modelos: Ofrece MLPClassifier para clasificación (binaria o multiclase) y MLPRegressor para predicción de valores numéricos continuos.

Algoritmos de optimización: Utiliza optimizadores como Adam (por defecto, ideal para datos grandes) y LBFGS (muy rápido y preciso para datos pequeños).

Importar librerías

In [14]:
import pandas as pd
import seaborn as sns
import numpy as np
import random

import matplotlib.pyplot as plt
from sklearn.datasets import load_iris #Conjunto de datos Iris
from sklearn.model_selection import train_test_split #Para partición del conjunto de datos en entrenamiento y prueba
from sklearn.neural_network import MLPClassifier #Se importa Multi Layer Perceptron para clasificación
from sklearn.preprocessing import LabelEncoder #Para convertir etiquetas
from sklearn.metrics import mean_absolute_error, mean_squared_error #Para validación con cálculo de errores en predicciones
from sklearn.metrics import ConfusionMatrixDisplay #Matriz de confusión (para validación)
from sklearn.model_selection import GridSearchCV

Cargar conjunto de datos

In [4]:
#Cargar conjunto de datos
iris = load_iris()

#Convertir a pandas (dataframe - df)
iris_features = pd.DataFrame(data=iris.data, columns=iris.feature_names) #Características X

Definir entradas como matriz X

In [5]:
X = iris_features.copy()
X.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


Codificar variables categóricas (nombres de especies) en variables numéricas (0, 1 o 2)

In [6]:
le = LabelEncoder()
encod = le.fit_transform(iris.target_names)
print(encod)

[0 1 2]


Partir el conjunto de datos en entrenamiento (66%) y validación (33%)

El parámetro "random_state" sirve para controlar la aleatoriedad y siempre, aunque aleatoria, hacer la partición de la misma manera, lo cual es útil para hacer la partición en diferentes simulaciones y con diferentes modelos con la misma partición.

In [7]:
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.33, random_state=101)
print('X_train: ', X_train.shape)
print('X_test: ', X_test.shape)
print('y_train: ', y_train.shape)
print('y_test: ', y_test.shape)

X_train:  (100, 4)
X_test:  (50, 4)
y_train:  (100,)
y_test:  (50,)


Modelo

Usamos una red neuronal base, sin modificación en hiperparámetros

In [9]:
nn = MLPClassifier()
nn.fit(X_train, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


MLPClassifier()

In [10]:
pred = nn.predict(X_test)
print(pred)

[0 0 0 2 1 2 1 1 2 0 2 0 0 2 2 1 1 1 0 2 1 0 1 1 1 1 1 2 0 0 2 1 2 1 2 1 1
 1 1 2 0 0 0 1 2 0 2 1 0 1]


In [11]:
print('MSE:', mean_squared_error(y_test, pred))

MSE: 0.02


Se obtuvo un error de 2%. Veamos si cambiando la arquitectura de la red en cuanto a sus capas ocultas y números de neuronas en cada capa, además probando 2 funciones de activación, se puede encontrar una mejor solución.

In [12]:
param_grid = {
    'hidden_layer_sizes': [
        (50,),               # 1 capa oculta con 50 neuronas
        (100,),              # 1 capa oculta con 100 neuronas
        (50, 50),            # 2 capas ocultas con 50 neuronas cada una
        (100, 50, 25)        # 3 capas ocultas decrecientes
    ],
    'activation': ['relu', 'tanh'], # Puedes añadir otros hiperparámetros si deseas
}

In [16]:
nn = MLPClassifier()
grid_search = GridSearchCV(estimator=nn, param_grid=param_grid, cv=3, n_jobs=-1, scoring='accuracy')

# 5. Entrenar explorando todas las combinaciones
grid_search.fit(X_train, y_train)

# 6. Resultados obtenidos
print(f"Mejor tamaño de capas y parámetros: {grid_search.best_params_}")
print(f"Mejor puntuación de precisión (Accuracy): {grid_search.best_score_:.4f}")

Mejor tamaño de capas y parámetros: {'activation': 'tanh', 'hidden_layer_sizes': (50, 50)}
Mejor puntuación de precisión (Accuracy): 0.9697


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [17]:
# Evaluar el mejor modelo en los datos de prueba
mejor_modelo = grid_search.best_estimator_
print(f"Precisión en el set de prueba: {mejor_modelo.score(X_test, y_test):.4f}")

Precisión en el set de prueba: 1.0000


¡Se logró una clasificación PERFECTA!

En este caso se encontró que una red neuronal con 50 neuronas en 2 capas ocultas, además con función de activación tanh, logra solucionar perfectamente este problema del conjunto de datos IRIS (o por lo menos con la partición aleatoria que se hizo entre entrenamiento y prueba), al clasificar perfectamente todas las muestras del subconjunto de validación.